In [1]:
import time
import requests
import pandas as pd

url = "https://servicodados.ibge.gov.br/api/v3/agregados/4093/periodos/201201|201202|201203|201204|201301|201302|201303|201304|201401|201402|201403|201404|201501|201502|201503|201504|201601|201602|201603|201604|201701|201702|201703|201704|201801|201802|201803|201804|201901|201902|201903|201904|202001|202002|202003|202004|202101|202102|202103|202104|202201|202202|202203|202204|202301|202302|202303|202304|202401|202402|202403|202404|202501|202502|202503|202504|202601|202602/variaveis/1641|4088|4090|4092|4094|4096|4097|4098|4099|12466|4723?localidades=N3[26]&classificacao=2[6794]"

def get_with_retry(url, max_retries=5, wait=2):
    for attempt in range(1, max_retries + 1):
        try:
            response = requests.get(url, timeout=10)

            if response.status_code == 200:
                return response

            print(f"Attempt {attempt}: got {response.status_code}")

        except requests.exceptions.RequestException as e:
            # Network errors, timeouts, connection refused, etc.
            print(f"Attempt {attempt}: {e}")

        time.sleep(wait * attempt)  # waits 2s, 4s, 6s... (simple backoff)

    raise RuntimeError(f"Failed after {max_retries} attempts: {url}")

response = get_with_retry(url)
data = response.json()


In [2]:
import json

with open("dados_desemprego.json",'w',encoding='UTF-8') as f:
    json.dump(data,f,ensure_ascii=False,indent=2)

with open("dados_desemprego.json",'r',encoding='UTF-8') as f:
    data = json.load(f)

In [3]:
total = data[0]["resultados"][0]["series"][0]["serie"]
forca = data[1]["resultados"][0]["series"][0]["serie"]
ocupadas = data[2]["resultados"][0]["series"][0]["serie"]
desocupadas = data[3]["resultados"][0]["series"][0]["serie"]
fora_da_forca = data[4]["resultados"][0]["series"][0]["serie"]
forca_taxa_participacao = data[5]["resultados"][0]["series"][0]["serie"]
nivel_de_ocupacao = data[6]["resultados"][0]["series"][0]["serie"]
nivel_de_desocupacao = data[7]["resultados"][0]["series"][0]["serie"]
taxa_de_desocupacao = data[8]["resultados"][0]["series"][0]["serie"]
taxa_de_informalidade = data[9]["resultados"][0]["series"][0]["serie"]
ocupadas_em_informalidade = data[10]["resultados"][0]["series"][0]["serie"]


In [4]:
# Completar
def JSON_to_df(series,series_name):
    df = pd.DataFrame.from_dict(data=series,orient='index',columns=[series_name])
    df.index.name = 'periodo'
    df = df.reset_index()
    df['ano'] = df['periodo'].str[:4]
    df['tri'] = df['periodo'].str[-1:]
    df['periodo'] = pd.PeriodIndex(df['ano']+'Q'+df['tri'], freq='Q')
    df = df.drop(labels=['ano','tri'],axis='columns')
    return df
    pass

df_total = JSON_to_df(total,"total")
df_forca = JSON_to_df(forca,"forca")
df_ocupadas = JSON_to_df(ocupadas,"ocupadas")
df_desocupadas = JSON_to_df(desocupadas,"desocupadas")
df_fora_da_forca = JSON_to_df(fora_da_forca,"fora_da_forca")
df_forca_taxa_participacao = JSON_to_df(forca_taxa_participacao,"forca_taxa_participacao")
df_nivel_de_ocupacao = JSON_to_df(nivel_de_ocupacao,"nivel_de_ocupacao")
df_nivel_de_desocupacao = JSON_to_df(nivel_de_desocupacao,"nivel_de_desocupacao")
df_taxa_de_desocupacao = JSON_to_df(taxa_de_desocupacao,"taxa_de_desocupacao")
df_taxa_de_informalidade = JSON_to_df(taxa_de_informalidade,"taxa_de_informalidade")
df_ocupadas_em_informalidade = JSON_to_df(ocupadas_em_informalidade,"ocupadas_em_informalidade")

In [5]:
df_ocupadas_em_informalidade

,periodo,ocupadas_em_informalidade
0,2012Q1,...
1,2012Q2,...
2,2012Q3,...
3,2012Q4,...
4,2013Q1,...
5,2013Q2,...
6,2013Q3,...
7,2013Q4,...
8,2014Q1,...
9,2014Q2,...


In [6]:
df = df_total.merge(df_forca,on="periodo",how="left")
df = df.merge(df_ocupadas,on="periodo",how="left")
df = df.merge(df_desocupadas,on="periodo",how="left")
df = df.merge(df_fora_da_forca,on="periodo",how="left")
df = df.merge(df_forca_taxa_participacao,on="periodo",how="left")
df = df.merge(df_nivel_de_ocupacao,on="periodo",how="left")
df = df.merge(df_nivel_de_desocupacao,on="periodo",how="left")
df = df.merge(df_taxa_de_desocupacao,on="periodo",how="left")
df  = df.merge(df_taxa_de_informalidade,on="periodo",how="left")
df = df.merge(df_ocupadas_em_informalidade,on="periodo",how="left")
df

,periodo,total,forca,ocupadas,desocupadas,fora_da_forca,forca_taxa_participacao,nivel_de_ocupacao,nivel_de_desocupacao,taxa_de_desocupacao,taxa_de_informalidade,ocupadas_em_informalidade
0,2012Q1,6857,3884,3512,372,2974,56.6,51.2,5.4,9.6,...,...
1,2012Q2,6853,3791,3477,314,3062,55.3,50.7,4.6,8.3,...,...
2,2012Q3,6894,3910,3543,367,2984,56.7,51.4,5.3,9.4,...,...
3,2012Q4,6942,3858,3503,355,3084,55.6,50.5,5.1,9.2,...,...
4,2013Q1,6936,3812,3405,407,3124,55.0,49.1,5.9,10.7,...,...
5,2013Q2,6983,3805,3436,369,3178,54.5,49.2,5.3,9.7,...,...
6,2013Q3,7007,3862,3535,327,3145,55.1,50.4,4.7,8.5,...,...
7,2013Q4,7035,3913,3623,290,3122,55.6,51.5,4.1,7.4,...,...
8,2014Q1,7040,3859,3519,340,3181,54.8,50.0,4.8,8.8,...,...
9,2014Q2,7048,3904,3592,313,3144,55.4,51.0,4.4,8.0,...,...


In [ ]:
df = df.rename(columns={'forca' : 'forca_de_trabalho'})

taxas = ['forca_taxa_participacao', 'nivel_de_ocupacao',
         'nivel_de_desocupacao', 'taxa_de_desocupacao', 'taxa_de_informalidade']
inteiros = ['total', 'forca_de_trabalho', 'ocupadas', 'desocupadas',
            'fora_da_forca', 'ocupadas_em_informalidade']

for col in taxas:
    df[col] = pd.to_numeric(df[col], errors="coerce")

for col in inteiros:
    df[col] = pd.to_numeric(df[col], errors="coerce").astype("Int64")

print(df.dtypes)

Index(['periodo', 'total', 'forca_de_trabalho', 'ocupadas', 'desocupadas',
       'fora_da_forca', 'forca_taxa_participacao', 'nivel_de_ocupacao',
       'nivel_de_desocupacao', 'taxa_de_desocupacao', 'taxa_de_informalidade',
       'ocupadas_em_informalidade'],
      dtype='str')

periodo                          str
total                          Int64
forca_de_trabalho              Int64
ocupadas                       Int64
desocupadas                    Int64
fora_da_forca                  Int64
forca_taxa_participacao      float64
nivel_de_ocupacao            float64
nivel_de_desocupacao         float64
taxa_de_desocupacao          float64
taxa_de_informalidade        float64
ocupadas_em_informalidade      Int64
dtype: object


In [20]:
for col in df.columns:
        if isinstance(df[col].dtype, pd.PeriodDtype):
            df[col] = df[col].astype(str)

print(df.dtypes)

periodo                      str
total                        str
forca_de_trabalho            str
ocupadas                     str
desocupadas                  str
fora_da_forca                str
forca_taxa_participacao      str
nivel_de_ocupacao            str
nivel_de_desocupacao         str
taxa_de_desocupacao          str
taxa_de_informalidade        str
ocupadas_em_informalidade    str
dtype: object


In [ ]:
subset = df.select_dtypes(include='')
subset

""
0
1
2
3
4
5
6
7
8
9
